<a href="https://colab.research.google.com/github/Mohamad-Selawy/Python/blob/master/Data%20Analysis/Level%202/groupby_and_pivot_tables.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **GroupBy and Pivot Tables**

_Use `groupby()` to split-apply-combine data, and `pivot_table()` to reshape and summarize_

## What You'll Learn
- How to use `groupby()` to compute grouped statistics
- How to create pivot tables for aggregated summaries
- Differences and use-cases for GroupBy vs Pivot Table



## Import Libraries

In [1]:
import pandas as pd
import numpy as np

## Sample Dataset

Here, a sample dataset is created and displayed using a Pandas DataFrame. This data will be used throughout the notebook to demonstrate `groupby()` and `pivot_table()` functionalities.

In [2]:
data = {
    'Department': ['Sales', 'Sales', 'HR', 'HR', 'IT', 'IT', 'Sales', 'Marketing', 'Marketing', 'IT', 'Finance', 'R&D', 'Support'],
    'Employee': ['Alice', 'Bob', 'Carol', 'David', 'Eve', 'Frank', 'Grace', 'Heidi', 'Ivan', 'Judy', 'Kevin', 'Liam', 'Mia'],
    'Salary': [50000, 55000, 48000, 52000, 60000, 61000, 53000, 58000, None, None, None, None, None],
    'Years_Experience': [2, 3, 4, 2, 5, 6, 3, 3, 2, 7, 7, 1, 1] # Liam (R&D, 1 year, None salary), Mia (Support, 1 year, None salary)
}

df = pd.DataFrame(data)
df

,Department,Employee,Salary,Years_Experience
0,Sales,Alice,50000.0,2
1,Sales,Bob,55000.0,3
2,HR,Carol,48000.0,4
3,HR,David,52000.0,2
4,IT,Eve,60000.0,5
5,IT,Frank,61000.0,6
6,Sales,Grace,53000.0,3
7,Marketing,Heidi,58000.0,3
8,Marketing,Ivan,NaN,2
9,IT,Judy,NaN,7


## 1. groupby() - Split, Apply, Combine

**Group by a single column:**

The `groupby()` method is fundamental for aggregating data. It involves three steps: **Split** (dividing data into groups based on some criteria), **Apply** (applying a function to each group independently), and **Combine** (combining the results into a data structure). This section demonstrates grouping by single and multiple columns, and applying multiple aggregation functions.

### Understanding `.agg()` vs. Direct Aggregation Methods

When performing aggregations after a `groupby()` operation in pandas, you have a couple of primary approaches: using direct aggregation methods (like `.mean()`, `.sum()`, `.max()`) or the more flexible `.agg()` method.

**1. Direct Aggregation Methods (`.mean()`, `.sum()`, `.max()`, etc.):**

*   **Simplicity:** These methods are straightforward and concise when you need to apply a single aggregation function to one or more columns.
*   **Syntax:** You typically select the column(s) you want to aggregate and then call the method directly on the `GroupBy` object.
*   **Output:** They return a Series if you aggregate a single column, or a DataFrame if you aggregate multiple columns (in which case the aggregation function is applied to each selected column).


In [3]:
df.groupby('Department')['Salary'].mean()

,Salary
Department,
Finance,NaN
HR,50000.000000
IT,60500.000000
Marketing,58000.000000
R&D,NaN
Sales,52666.666667
Support,NaN


**Group by multiple columns:**

In [4]:
df.groupby(['Department', 'Years_Experience'])['Salary'].mean()

Department  Years_Experience
Finance     7                       NaN
HR          2                   52000.0
            4                   48000.0
IT          5                   60000.0
            6                   61000.0
            7                       NaN
Marketing   2                       NaN
            3                   58000.0
R&D         1                       NaN
Sales       2                   50000.0
            3                   54000.0
Support     1                       NaN
Name: Salary, dtype: float64

**Apply multiple aggregation functions:**

**2. The `.agg()` Method (Aggregate):**

*   **Flexibility:** This is the main strength of `.agg()`. It allows for much more complex and varied aggregations:
    *   **Multiple functions on one column:** Apply several aggregation functions (e.g., 'mean', 'max', 'min') to a single column.
    *   **Renaming output columns:** You can provide custom names for the aggregated columns for better readability.

*   **Syntax:** `.agg()` offers several syntaxes:
    *   **Dictionary format:** This is highly versatile, where keys are column names and values are either a single aggregation function (passed as a string or a callable) or a list of functions.
    *   **Named aggregation (keyword arguments with tuples):** This is a very readable way to apply multiple aggregations and rename columns simultaneously. You pass keyword arguments directly to `.agg()`, where the keyword becomes the new column name, and the value is a tuple `('original_column', 'aggregation_function')`.
    

*   **Output:** Always returns a DataFrame, providing a consistent structure even for single aggregations.

**In summary:**

*   Choose **direct methods** (`.mean()`, `.sum()`) for quick, single-type aggregations where conciseness is preferred.
*   Opt for **`.agg()`** when you need more control, such as applying multiple or different aggregation functions across various columns, using custom functions, or custom-naming your output. It provides a powerful and flexible way to summarize your grouped data.

In [5]:
df.groupby('Department').agg({
    'Salary': ['mean', 'max', 'min'],
    'Years_Experience': 'median'
})

Salary                   Years_Experience
                    mean      max      min           median
Department                                                 
Finance              NaN      NaN      NaN              7.0
HR          50000.000000  52000.0  48000.0              3.0
IT          60500.000000  61000.0  60000.0              6.0
Marketing   58000.000000  58000.0  58000.0              2.5
R&D                  NaN      NaN      NaN              1.0
Sales       52666.666667  55000.0  50000.0              3.0
Support              NaN      NaN      NaN              1.0

In [6]:
df.groupby('Department').agg(
    mean_salary=('Salary', 'mean'),
    max_salary=('Salary', 'max'),
    median_years_experience=('Years_Experience', 'median')
).reset_index()


,Department,mean_salary,max_salary,median_years_experience
0,Finance,NaN,NaN,7.0
1,HR,50000.000000,52000.0,3.0
2,IT,60500.000000,61000.0,6.0
3,Marketing,58000.000000,58000.0,2.5
4,R&D,NaN,NaN,1.0
5,Sales,52666.666667,55000.0,3.0
6,Support,NaN,NaN,1.0


## 2. pivot_table() - Flexible Table Generator

The `pivot_table()` function is used to create spreadsheet-style pivot tables as a DataFrame. It summarizes data by specifying `index`, `columns`, `values`, and `aggfunc`. This section illustrates how to create pivot tables with single and multiple value aggregations.

The `pd.pivot_table()` function is a powerful tool in pandas for summarizing and reshaping data. It creates a spreadsheet-style pivot table as a DataFrame.

Here's a breakdown of its main arguments:

*   **`data`**: (Required) The DataFrame you want to pivot.
*   **`values`**: (Optional) The column or columns to aggregate.
*   **`index`**: (Optional) The column(s) to use to form the new DataFrame's index. These will be the rows of your pivot table.
*   **`columns`**: (Optional) The column(s) to use to form the new DataFrame's columns. These will be the columns of your pivot table.
*   **`aggfunc`**: (Optional) The aggregation function(s) to apply. This can be a function (e.g., `len`), a string (e.g., 'sum', 'mean', 'count'), or a list/dict of functions/strings. If a list, multiple aggregation functions will be applied to the `values`. If a dictionary, it allows different aggregation functions for different `values` columns.
*   **`fill_value`**: (Optional) Value to replace missing values (NaN) in the pivot table with. This is useful for presentation.
*   **`margins`**: (Optional) If `True`, adds all row/column totals (e.g., 'All' for `index` and `columns`).

In [12]:
pd.pivot_table(df,
               values='Salary',
               index='Department',
               columns='Years_Experience',
               aggfunc='mean')


Years_Experience,2,3,4,5,6
Department,,,,,
HR,52000.0,NaN,48000.0,NaN,NaN
IT,NaN,NaN,NaN,60000.0,61000.0
Marketing,NaN,58000.0,NaN,NaN,NaN
Sales,50000.0,54000.0,NaN,NaN,NaN


In [8]:
pd.pivot_table(df,
               values='Salary',
               index='Department',
               columns='Years_Experience',
               aggfunc='mean',
               fill_value=0,
               margins=True) # Adds all row/column totals


Years_Experience,2,3,4,5,6,All
Department,,,,,,
HR,52000.0,0.000000,48000.0,0.0,0.0,50000.000000
IT,0.0,0.000000,0.0,60000.0,61000.0,60500.000000
Marketing,0.0,58000.000000,0.0,0.0,0.0,58000.000000
Sales,50000.0,54000.000000,0.0,0.0,0.0,52666.666667
All,51000.0,55333.333333,48000.0,60000.0,61000.0,54625.000000


**Pivot table with multiple values:**

In [9]:
pd.pivot_table(df,
               values=['Salary', 'Years_Experience'],
               index='Department',
               aggfunc={'Salary': ['mean', 'max'], 'Years_Experience': 'max'},
               fill_value=0)

Salary               Years_Experience
                max          mean              max
Department                                        
Finance         0.0      0.000000                7
HR          52000.0  50000.000000                4
IT          61000.0  60500.000000                7
Marketing   58000.0  58000.000000                3
R&D             0.0      0.000000                1
Sales       55000.0  52666.666667                3
Support         0.0      0.000000                1

## Key Differences

| Feature       | `groupby()`                  | `pivot_table()`                            |
|:------------- |:---------------------------- |:------------------------------------------ |
| Use Case      | Grouping + aggregation logic | Tabular summarization                      |
| Output Format | Series or DataFrame          | Matrix-style DataFrame                     |
| Flexibility   | High                         | Higher for tables (index/columns/aggfuncs) |
| Null Handling | Keeps NaNs                   | Can handle NaNs using `fill_value`         |


## Practice Exercise
Try this on your own:

```
# Sample challenge:
# Create a pivot table showing mean salary by Department and Years_Experience
```

## Summary
- `groupby()` is great for programmatic analysis
- `pivot_table()` is ideal for presentation-ready summaries



This section provides a clear comparison between `groupby()` and `pivot_table()`, highlighting their primary use cases, output formats, flexibility, and null handling capabilities.